# Does flat-field correction (FFC) improve DAPI registration at FOV borders?

`test_dapi_edge_filter.ipynb`'s DoG-filtered DAPI image showed clear
cell/nucleus edges at the frame CENTER but much less clear edges toward
the frame BORDER -- raising the question of whether this is a real,
FFC-fixable radiometric artifact (vignetting: less illumination/collection
efficiency off-axis, a genuine photon-count deficit) or a geometric one
(off-axis optical aberrations blurring the PSF, which no radiometric
correction touches). This matters directly for stitching: every
registration crop this pipeline ever computes (`crop_overlap`) is drawn
from the OVERLAP BAND between adjoining tiles -- which, by construction,
sits at each FOV's own edge. So whatever border degradation exists hits
exactly the pixels registration depends on, not an incidental corner case.

**Confirmed before building this**: the current stitching/registration
pipeline reads raw, non-flat-fielded frames -- `MERci.analysis.ffc`'s own
docstring states FFC only affects the round-mosaic path
(`MERci.analysis.round`), never the per-FOV registration pipeline used
here or in `test_stitching.ipynb`.

**A key methodological subtlety, addressed explicitly in this notebook's
own design**: a self-shift test (one real image vs. a synthetic shift of
ITSELF) can be fooled here in a way it couldn't be for the DAPI-inversion/
edge-filter notebooks. FFC divides by a field that is <1 near the border,
so it doesn't just restore attenuated real signal -- it also amplifies
whatever NOISE was already there. Amplified, spatially-uncorrelated shot
noise can itself produce a sharp-looking self-shift decay (its own
autocorrelation is a near-delta-function), which would look identical to
"real sharpened structure" in a naive read of that one test. The fix:
independent noise in two SEPARATELY acquired real images never correlates
with the other's independent noise, so the **cross-tile test** (and the
practical whole-grid test built on it) cannot be fooled this way -- those
are treated as the decisive evidence here; the self-shift/border-crop
section is kept for illustration only, with this caveat repeated at its
own display cell.

**Real FFC field**: computed via this repo's own production functions
(`MERci.analysis.ffc.select_ffc_exterior_fovs` + `compute_ffc_field_for_color`,
the `"exterior_grid"` strategy), for DAPI (405nm, same z-plane used
throughout this investigation) on `BC555_sample_05/disk` -- no field was
already cached for this color/dataset (the round-mosaics cache only has
560nm/650nm), so it is computed fresh here and cached under this
notebook's own `analysis/cache/`.

**disk dataset only**, continuing this investigation's own scope.

Follows `NOTEBOOK_GUIDELINES.md`: calculation cells cached under
`analysis/cache/test_ffc_border_registration/`, skip recomputation when a
valid cache exists, `ProgressReporter` progress, explicit plot font sizes,
every displayed figure also saved to `analysis/figures/`.

## 1 — Setup

In [ ]:
import os
import sys
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import shift as ndi_shift
from scipy.spatial import KDTree
from skimage.filters import sobel, difference_of_gaussians

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as this investigation's other notebooks.
MERCI_DIR = Path(os.getcwd()).parent.parent.parent   # MERci/

sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.io              import load_positions, read_image_frames
from MERci.common.experiment_info import load_experiment_info, resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs    import (
    find_frame_table_for_hal_config, get_camera_pixel_size_um, get_camera_frame_size,
    get_all_color_frame_indices,
)
from MERci.acquisition.positions       import find_grid_neighbor
from MERci.acquisition.camera_rotation import (
    apply_microscope_orientation, crop_overlap, register_neighbor_pair, overlap_correlation,
)
from MERci.acquisition.alignment  import remove_hot_pixels, phase_drift
from MERci.acquisition.configs import load_microscope_orientation
from MERci.analysis.ffc           import select_ffc_exterior_fovs, compute_ffc_field_for_color, apply_ffc, save_ffc_field, load_ffc_field
from MERci.progress_display       import ProgressReporter
from MERci.plots.experiment_plots import get_merci_figures_dir

NOTEBOOK_NAME = "test_ffc_border_registration"
print(f"MERCI_DIR : {MERCI_DIR}")

## 2 — Parameters

In [ ]:
DATASETS = [
    ("disk", "/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/breast_cancer/experiments/BC555_sample_05/disk"),
]
IMAGE_SUFFIX = ".zarr"

BEAD_FRAME_INDEX = 0     # unused here (DAPI-only investigation) but kept for parity with sibling notebooks
DAPI_COLOR_NM    = 405.0
DAPI_Z_INDEX     = 3     # same z-index every notebook in this investigation uses

CHANNELS = ("dapi_raw", "dapi_ffc")
CHANNEL_COLORS = {"dapi_raw": "tab:orange", "dapi_ffc": "tab:purple"}

FFC_SMOOTH_SIGMA_PX = 50.0   # this repo's own compute_ffc_field_for_color default

# Sanity-check DoG params, same values test_dapi_edge_filter.ipynb used
# (kept comparable, not re-tuned here).
DOG_LOW_SIGMA  = 1.0
DOG_HIGH_SIGMA = 8.0

# Border-crop direction used for the self-shift comparison -- "up" is
# arbitrary but representative; crop_overlap(img, img, "up", overlap_fraction)[0]
# reuses the exact overlap-band geometry every real registration call uses.
BORDER_DIRECTION = "up"

N_RADIAL_FOVS = 10   # real FOVs averaged for the radial gradient-energy curve
N_RADIAL_BINS = 30

TOLERANCE_FRACTION = 0.25
UPSAMPLE_FACTOR     = 10
SEED = 0

SHIFT_RANGE_PX = np.arange(-3.0, 3.01, 0.25)
N_EDGES_FOR_SHIFT_CURVE = 8

N_ANCHORS = 20
ANCHOR_BATCH_SIZE = 5   # disk-proven memory-safe batch size (see this investigation's other notebooks)

FORCE_RECOMPUTE = False

PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 11
PLOT_LEGEND_FONTSIZE = 10

print(f"Datasets: {[label for label, _ in DATASETS]}")

## 3 — Resolve the cells round + FOV geometry (per dataset)

Same as this investigation's other notebooks, extended to also return
`meta` and `cells_round_id` -- needed here to call
`select_ffc_exterior_fovs`.

In [ ]:
def resolve_dataset_geometry(dataset_label, dataset_dir):
    sample_dir = Path(dataset_dir)
    sample_name, imaging_dir = resolve_sample_identity(sample_dir / "MERci")
    positions_tag = positions_file_tag(sample_name, imaging_dir)

    info       = load_experiment_info(sample_dir / "metadata" / "experiment_info.yaml")
    microscope = info.microscope

    config = ExperimentConfig.from_sample_dir(
        sample_dir,
        positions_txt  = sample_dir / "positions" / f"positions_{positions_tag}.txt",
        image_suffix   = IMAGE_SUFFIX,
        microscope     = microscope,
    )
    meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                    image_suffix=config.image_suffix)

    cells_round_id = meta.round_for_imaging_type("cells")
    if not meta.round_fully_written(cells_round_id):
        print(f"WARNING [{dataset_label}]: cells round {cells_round_id} is not yet fully written on disk -- "
              f"some FOVs sampled below may be missing.")

    cells_series = next(s for s in meta.series_for_round(cells_round_id) if s.hal_config)
    hal_path     = Path(config.settings_dir) / cells_series.hal_config
    ft_path      = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
    if ft_path is None or not ft_path.exists():
        raise FileNotFoundError(f"No frame table found for the cells round (hal_config={hal_path}).")
    frame_table = pd.read_csv(ft_path, index_col=0)

    dapi_frame_indices = get_all_color_frame_indices(frame_table, DAPI_COLOR_NM)
    if DAPI_Z_INDEX >= len(dapi_frame_indices):
        raise ValueError(
            f"[{dataset_label}] DAPI_Z_INDEX={DAPI_Z_INDEX} out of range -- {DAPI_COLOR_NM:.0f}nm only has "
            f"{len(dapi_frame_indices)} z-plane(s) in this frame table.")
    dapi_frame_index = dapi_frame_indices[DAPI_Z_INDEX]

    pixel_size_um      = get_camera_pixel_size_um(microscope)
    frame_width_px, _  = get_camera_frame_size(microscope)
    frame_width_um     = frame_width_px * pixel_size_um

    full_positions = load_positions(config.positions_txt)
    cells_fov_ids  = sorted(f for f in full_positions if f in meta.fovs)

    coords_arr = np.array([full_positions[f] for f in cells_fov_ids], dtype=float)
    nn_dist, _ = KDTree(coords_arr).query(coords_arr, k=2)
    step_size_um     = float(np.median(nn_dist[:, 1]))
    overlap_fraction = max(0.0, 1.0 - step_size_um / frame_width_um)

    orientation = load_microscope_orientation(microscope, MERCI_DIR / "data" / "configs" / "merlin" / "microscope")
    orient_transpose       = bool(orientation.get("transpose", False))
    orient_flip_horizontal = bool(orientation.get("flip_horizontal", False))
    orient_flip_vertical   = bool(orientation.get("flip_vertical", False))

    cache_dir   = config.analysis_dir / "cache" / NOTEBOOK_NAME
    figures_dir = get_merci_figures_dir(sample_dir, "tests", NOTEBOOK_NAME, subfolder="fov_stitching")
    cache_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    print(f"[{dataset_label}] microscope={microscope}  cells_round={cells_round_id}  "
          f"dapi_frame={dapi_frame_index}  pixel_size_um={pixel_size_um}  "
          f"step_size_um={step_size_um:.3f}  overlap_fraction={overlap_fraction:.3f}  "
          f"n_fovs={len(cells_fov_ids)}")
    print(f"[{dataset_label}] orientation: transpose={orient_transpose}  "
          f"flip_horizontal={orient_flip_horizontal}  flip_vertical={orient_flip_vertical}")

    return {
        "dataset_label": dataset_label, "sample_dir": sample_dir, "config": config, "meta": meta,
        "cells_round_id": cells_round_id, "cells_series": cells_series, "dapi_frame_index": dapi_frame_index,
        "pixel_size_um": pixel_size_um, "frame_width_um": frame_width_um,
        "full_positions": full_positions, "cells_fov_ids": cells_fov_ids,
        "step_size_um": step_size_um, "overlap_fraction": overlap_fraction,
        "orient_transpose": orient_transpose, "orient_flip_horizontal": orient_flip_horizontal,
        "orient_flip_vertical": orient_flip_vertical,
        "cache_dir": cache_dir, "figures_dir": figures_dir,
    }

## 4 — Compute (or load) the real FFC field, + 2-channel frame loader

Same production call `MERci.scheduler.build_round_mosaics` would make for
this color: `select_ffc_exterior_fovs` (every FOV on the imaged grid's
own exterior, at the DAPI z-plane) feeds `compute_ffc_field_for_color`
(running-mean over samples -> `smooth_sigma_px`-Gaussian-smooth ->
normalize -> floor-clip). Cached under this notebook's own
`analysis/cache/` so it is computed once.

In [ ]:
def resolve_ffc_field(dataset_geo):
    field_path = dataset_geo["cache_dir"] / "ffc_field_dapi_405nm.npz"
    if not FORCE_RECOMPUTE and field_path.exists():
        field, meta = load_ffc_field(field_path)
        print(f"[{dataset_geo['dataset_label']}] Loaded cached FFC field from {field_path} "
              f"(n_samples={meta.get('n_samples')})")
        return field

    samples = select_ffc_exterior_fovs(
        dataset_geo["cells_round_id"], dataset_geo["config"], dataset_geo["meta"], dataset_geo["dapi_frame_index"],
    )
    print(f"[{dataset_geo['dataset_label']}] Estimating FFC field from {len(samples)} exterior-FOV samples...")
    field, field_meta = compute_ffc_field_for_color(
        samples, frame_width=dataset_geo["config"].frame_width, frame_height=dataset_geo["config"].frame_height,
        smooth_sigma_px=FFC_SMOOTH_SIGMA_PX,
    )
    save_ffc_field(field_path, field, field_meta)
    print(f"[{dataset_geo['dataset_label']}] Saved FFC field to {field_path} -- "
          f"range [{field.min():.3f}, {field.max():.3f}]")
    return field


def make_channel_loaders(dataset_geo, ffc_field):
    """Returns (load_frame_for_channel, raw_cache) -- raw_cache holds only
    the real DAPI read (uint16); dapi_ffc is a cheap on-demand division,
    not cached a second time (same memory-safety reasoning as this
    investigation's other notebooks)."""
    cells_series, config = dataset_geo["cells_series"], dataset_geo["config"]
    dapi_frame_index = dataset_geo["dapi_frame_index"]
    ot, ofh, ofv = dataset_geo["orient_transpose"], dataset_geo["orient_flip_horizontal"], dataset_geo["orient_flip_vertical"]

    raw_cache = {}   # {fov_id: dapi_raw}, oriented uint16

    def _get_raw(fov_id):
        if fov_id not in raw_cache:
            path = cells_series.resolve_path(fov_id, config.image_suffix)
            (dapi_raw,) = read_image_frames(path, [dapi_frame_index],
                                             frame_width=config.frame_width, frame_height=config.frame_height)
            raw_cache[fov_id] = apply_microscope_orientation(dapi_raw, transpose=ot, flip_horizontal=ofh, flip_vertical=ofv)
        return raw_cache[fov_id]

    def load_frame_for_channel(channel):
        def _load(fov_id):
            dapi_raw = _get_raw(fov_id)
            if channel == "dapi_raw":
                return dapi_raw
            return apply_ffc(dapi_raw, ffc_field)
        return _load

    return load_frame_for_channel, raw_cache


RESULTS = {}
for dataset_label, dataset_dir in DATASETS:
    print(f"\n=== {dataset_label} ({dataset_dir}) ===")
    dataset_geo = resolve_dataset_geometry(dataset_label, dataset_dir)
    ffc_field = resolve_ffc_field(dataset_geo)
    load_frame_for_channel, raw_cache = make_channel_loaders(dataset_geo, ffc_field)
    RESULTS[dataset_label] = {
        "dataset_geo": dataset_geo, "ffc_field": ffc_field,
        "load_frame_for_channel": load_frame_for_channel, "raw_cache": raw_cache,
    }
print(f"\nDatasets resolved: {list(RESULTS.keys())}")

## 5 — Radial gradient-energy test (calculation)

Does FFC recover real high-spatial-frequency content toward the frame
edge, or does it only rescale the mean level (leaving the true
`integral((dI/dx)^2)` -- the Cramer-Rao-relevant quantity from this
investigation's earlier discussion -- essentially unchanged, or even
noisier)? Sobel gradient-magnitude-squared, binned by radial distance
from the frame's own optical center, averaged over `N_RADIAL_FOVS` real
FOVs.

In [ ]:
def compute_radial_gradient_energy(dataset_geo, load_frame_for_channel):
    cache_csv = dataset_geo["cache_dir"] / "radial_gradient_energy.csv"
    if not FORCE_RECOMPUTE and cache_csv.exists():
        df = pd.read_csv(cache_csv)
        print(f"[{dataset_geo['dataset_label']}] Loaded cached radial gradient energy from {cache_csv}")
        return df

    rng = np.random.default_rng(SEED)
    candidates = list(dataset_geo["cells_fov_ids"])
    rng.shuffle(candidates)
    sample_fovs = candidates[:N_RADIAL_FOVS]

    h, w = load_frame_for_channel(CHANNELS[0])(sample_fovs[0]).shape   # actual frame shape, config.frame_* may be unset
    yy, xx = np.mgrid[0:h, 0:w]
    radius = np.hypot(yy - h / 2.0, xx - w / 2.0)
    max_radius = radius.max()
    bin_edges = np.linspace(0, max_radius, N_RADIAL_BINS + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_idx = np.clip(np.digitize(radius.ravel(), bin_edges) - 1, 0, N_RADIAL_BINS - 1)

    reporter = ProgressReporter(total=len(sample_fovs) * len(CHANNELS),
                                 label=f"[{dataset_geo['dataset_label']}] Radial gradient energy")
    rows = []
    for fov_id in sample_fovs:
        for channel in CHANNELS:
            img = remove_hot_pixels(load_frame_for_channel(channel)(fov_id)).astype(np.float64)
            grad_energy = sobel(img).ravel() ** 2
            binned_mean = np.bincount(bin_idx, weights=grad_energy, minlength=N_RADIAL_BINS) / \
                          np.maximum(np.bincount(bin_idx, minlength=N_RADIAL_BINS), 1)
            for b, val in zip(bin_centers, binned_mean):
                rows.append({"fov_id": fov_id, "channel": channel, "radius_px": float(b), "grad_energy": float(val)})
            reporter.update(1)
    reporter.done()

    df = pd.DataFrame(rows)
    df.to_csv(cache_csv, index=False)
    print(f"[{dataset_geo['dataset_label']}] Saved {len(df)} rows to {cache_csv}")
    return df


for dataset_label, R in RESULTS.items():
    R["radial_df"] = compute_radial_gradient_energy(R["dataset_geo"], R["load_frame_for_channel"])

## 6 — Radial gradient-energy curves (display)

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(7 * len(DATASETS), 6), squeeze=False)
for col, (dataset_label, R) in enumerate(RESULTS.items()):
    ax = axes[0, col]
    df = R["radial_df"]
    for channel in CHANNELS:
        sub = df[df["channel"] == channel]
        grouped = sub.groupby("radius_px")["grad_energy"]
        mean, std = grouped.mean(), grouped.std()
        ax.plot(mean.index, mean.values, label=channel, color=CHANNEL_COLORS[channel], linewidth=2)
        ax.fill_between(mean.index, mean.values - std.values, mean.values + std.values,
                         color=CHANNEL_COLORS[channel], alpha=0.15)
    ax.set_xlabel("Distance from frame center (px)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("Mean Sobel gradient energy (per pixel)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{dataset_label}: radial gradient energy, {N_RADIAL_FOVS} real FOVs", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()
fig.savefig(RESULTS["disk"]["dataset_geo"]["figures_dir"] / f"{NOTEBOOK_NAME}.radial_gradient_energy.png", dpi=150)
plt.show()

print("Gradient energy at center vs. edge (mean +/- std over FOVs, innermost vs. outermost radial bin):")
for dataset_label, R in RESULTS.items():
    df = R["radial_df"]
    inner_r, outer_r = df["radius_px"].min(), df["radius_px"].max()
    for channel in CHANNELS:
        sub = df[df["channel"] == channel]
        inner = sub[np.isclose(sub["radius_px"], inner_r)]["grad_energy"]
        outer = sub[np.isclose(sub["radius_px"], outer_r)]["grad_energy"]
        print(f"[{dataset_label}] {channel:10s}: center={inner.mean():.2f}+/-{inner.std():.2f}  "
              f"edge={outer.mean():.2f}+/-{outer.std():.2f}  ratio(edge/center)={outer.mean()/inner.mean():.3f}")

## 7 — Border-crop self-shift test (calculation)

**Illustrative only, not decisive** -- see this notebook's own intro
markdown: FFC can amplify independent per-acquisition noise into a
sharp-looking self-shift decay, which would look like "real sharpening"
here even if it is not. `crop_overlap(img, img, BORDER_DIRECTION,
overlap_fraction)[0]` extracts one real border-region strip using the
IDENTICAL geometry every registration call in this pipeline already
uses; the whole-frame curve is computed alongside it for direct
before/after-FFC and border-vs-center reference.

In [ ]:
def compute_self_shift_curve(dataset_geo, load_frame_for_channel):
    cache_csv = dataset_geo["cache_dir"] / "self_shift_curve.csv"
    if not FORCE_RECOMPUTE and cache_csv.exists():
        df = pd.read_csv(cache_csv)
        print(f"[{dataset_geo['dataset_label']}] Loaded cached self-shift curve from {cache_csv}")
        return df

    fov_id = dataset_geo["cells_fov_ids"][len(dataset_geo["cells_fov_ids"]) // 2]
    overlap_fraction = dataset_geo["overlap_fraction"]
    rows = []
    for channel in CHANNELS:
        full_img = remove_hot_pixels(load_frame_for_channel(channel)(fov_id)).astype(np.float64)
        border_img = remove_hot_pixels(
            crop_overlap(load_frame_for_channel(channel)(fov_id), load_frame_for_channel(channel)(fov_id),
                         BORDER_DIRECTION, overlap_fraction)[0]
        ).astype(np.float64)
        for region_name, img in (("whole_frame", full_img), ("border_crop", border_img)):
            for shift_px in SHIFT_RANGE_PX:
                shifted = ndi_shift(img, shift=(0.0, float(shift_px)), order=1, mode="nearest")
                corr = float(np.corrcoef(img.ravel(), shifted.ravel())[0, 1])
                rows.append({"fov_id": fov_id, "channel": channel, "region": region_name,
                             "shift_px": float(shift_px), "correlation": corr})

    df = pd.DataFrame(rows)
    df.to_csv(cache_csv, index=False)
    print(f"[{dataset_geo['dataset_label']}] Saved self-shift curve (FOV {fov_id}) to {cache_csv}")
    return df


for dataset_label, R in RESULTS.items():
    R["self_shift_df"] = compute_self_shift_curve(R["dataset_geo"], R["load_frame_for_channel"])

## 8 — Border-crop self-shift curves (display)

In [ ]:
fig, axes = plt.subplots(len(DATASETS), 2, figsize=(14, 6 * len(DATASETS)), squeeze=False)
for row, (dataset_label, R) in enumerate(RESULTS.items()):
    df = R["self_shift_df"]
    for col, region in enumerate(("whole_frame", "border_crop")):
        ax = axes[row, col]
        sub_region = df[df["region"] == region]
        for channel in CHANNELS:
            sub = sub_region[sub_region["channel"] == channel].sort_values("shift_px")
            ax.plot(sub["shift_px"], sub["correlation"], label=channel, color=CHANNEL_COLORS[channel], linewidth=2)
        ax.axhline(1.0, color="gray", linestyle=":", linewidth=1)
        ax.axvline(0.0, color="gray", linestyle="--", linewidth=1)
        ax.set_xlabel("Synthetic shift (px)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.set_ylabel("Pearson correlation vs. unshifted self", fontsize=PLOT_LABEL_FONTSIZE)
        ax.set_title(f"{dataset_label}: {region} (illustrative only -- see Section 7 caveat)",
                     fontsize=PLOT_TITLE_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
        ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()
fig.savefig(RESULTS["disk"]["dataset_geo"]["figures_dir"] / f"{NOTEBOOK_NAME}.self_shift_curves.png", dpi=150)
plt.show()

## 9 — Cross-tile test: real edges (calculation, decisive)

Two independently-acquired real images never share the SAME noise
realization, so amplified noise cannot inflate this metric the way it
could Section 8's self-shift test -- this is the first decisive test of
whether FFC helps. Same real-edge methodology as this investigation's
other notebooks: each edge+channel's own real best-alignment shift
(`phase_drift`) anchors the synthetic sweep.

In [ ]:
def compute_edge_shift_curves(dataset_geo, load_frame_for_channel):
    cache_csv = dataset_geo["cache_dir"] / "edge_shift_curves.csv"
    if not FORCE_RECOMPUTE and cache_csv.exists():
        df = pd.read_csv(cache_csv)
        print(f"[{dataset_geo['dataset_label']}] Loaded cached edge shift curves from {cache_csv}")
        return df

    full_positions   = dataset_geo["full_positions"]
    step_size_um     = dataset_geo["step_size_um"]
    overlap_fraction = dataset_geo["overlap_fraction"]

    rng = np.random.default_rng(SEED)
    candidates = list(dataset_geo["cells_fov_ids"])
    rng.shuffle(candidates)

    sampled_edges = []
    for anchor_fov in candidates:
        for direction in ("up", "down", "left", "right"):
            neighbor_fov = find_grid_neighbor(anchor_fov, full_positions, direction, step_size_um, TOLERANCE_FRACTION)
            if neighbor_fov is not None:
                sampled_edges.append((anchor_fov, neighbor_fov, direction))
        if len(sampled_edges) >= N_EDGES_FOR_SHIFT_CURVE:
            break
    sampled_edges = sampled_edges[:N_EDGES_FOR_SHIFT_CURVE]
    print(f"[{dataset_geo['dataset_label']}] Sampled {len(sampled_edges)} real edges: {sampled_edges}")

    reporter = ProgressReporter(total=len(sampled_edges) * len(CHANNELS) * len(SHIFT_RANGE_PX),
                                 label=f"[{dataset_geo['dataset_label']}] Computing edge shift curves")
    rows = []
    for edge_id, (anchor_fov, neighbor_fov, direction) in enumerate(sampled_edges):
        shift_axis = "col" if direction in ("left", "right") else "row"
        for channel in CHANNELS:
            anchor_img   = load_frame_for_channel(channel)(anchor_fov)
            neighbor_img = load_frame_for_channel(channel)(neighbor_fov)
            a_crop, n_crop = crop_overlap(anchor_img, neighbor_img, direction, overlap_fraction)
            a_crop = remove_hot_pixels(a_crop).astype(np.float64)
            n_crop = remove_hot_pixels(n_crop).astype(np.float64)

            base_shift, base_error = phase_drift(a_crop, n_crop, UPSAMPLE_FACTOR)
            base_dy, base_dx = float(base_shift[0]), float(base_shift[1])

            for shift_px in SHIFT_RANGE_PX:
                delta = (0.0, float(shift_px)) if shift_axis == "col" else (float(shift_px), 0.0)
                shift_tuple = (base_dy + delta[0], base_dx + delta[1])
                shifted = ndi_shift(n_crop, shift=shift_tuple, order=1, mode="nearest")
                a_flat, s_flat = a_crop.ravel(), shifted.ravel()
                corr = 0.0 if a_flat.std() == 0.0 or s_flat.std() == 0.0 else float(np.corrcoef(a_flat, s_flat)[0, 1])
                rows.append({"edge_id": edge_id, "anchor_fov": anchor_fov, "neighbor_fov": neighbor_fov,
                             "direction": direction, "channel": channel, "shift_px": float(shift_px),
                             "correlation": corr, "base_dy_px": base_dy, "base_dx_px": base_dx,
                             "base_registration_error": base_error})
                reporter.update(1)
    reporter.done()

    df = pd.DataFrame(rows)
    df.to_csv(cache_csv, index=False)
    print(f"[{dataset_geo['dataset_label']}] Saved {len(df)} rows to {cache_csv}")
    return df


for dataset_label, R in RESULTS.items():
    R["edge_shift_df"] = compute_edge_shift_curves(R["dataset_geo"], R["load_frame_for_channel"])

## 10 — Cross-tile curves: mean +/- std over 8 edges (display)

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(7 * len(DATASETS), 6), squeeze=False)
for col, (dataset_label, R) in enumerate(RESULTS.items()):
    ax = axes[0, col]
    df = R["edge_shift_df"]
    for channel in CHANNELS:
        sub = df[df["channel"] == channel]
        grouped = sub.groupby("shift_px")["correlation"]
        mean, std = grouped.mean(), grouped.std()
        ax.plot(mean.index, mean.values, label=channel, color=CHANNEL_COLORS[channel], linewidth=2)
        ax.fill_between(mean.index, mean.values - std.values, mean.values + std.values,
                         color=CHANNEL_COLORS[channel], alpha=0.15)
    ax.axvline(0.0, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("Synthetic shift beyond the real measured alignment (px)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("Pearson correlation vs. anchor crop", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{dataset_label}: mean +/- std over {df['edge_id'].nunique()} edges", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()
fig.savefig(RESULTS["disk"]["dataset_geo"]["figures_dir"] / f"{NOTEBOOK_NAME}.edge_shift_curves.png", dpi=150)
plt.show()

print("Sharpness proxy (correlation drop from shift=0 to |shift|=1px, mean over edges):")
for dataset_label, R in RESULTS.items():
    df = R["edge_shift_df"]
    print(f"[{dataset_label}]")
    for channel in CHANNELS:
        sub = df[df["channel"] == channel]
        at0 = sub[np.isclose(sub["shift_px"], 0.0)].groupby("edge_id")["correlation"].mean()
        at1 = sub[np.isclose(sub["shift_px"].abs(), 1.0)].groupby("edge_id")["correlation"].mean()
        drop = (at0 - at1).mean()
        print(f"  {channel:10s}: corr(0)={at0.mean():.4f}  corr(|1px|)={at1.mean():.4f}  drop={drop:.4f}")

## 11 — Practical test: real neighbor-tile registration, whole-grid sample (calculation)

Same disk-proven batched `register_neighbor_pair` replay this
investigation's other notebooks used to avoid the repeated OOM kills that
one-shot `sample_neighbor_correspondences` calls hit on this grid size.

In [ ]:
def compute_correspondences(dataset_geo, load_frame_for_channel, raw_cache):
    cache_dir        = dataset_geo["cache_dir"]
    full_positions   = dataset_geo["full_positions"]
    overlap_fraction = dataset_geo["overlap_fraction"]
    pixel_size_um    = dataset_geo["pixel_size_um"]
    step_size_um     = dataset_geo["step_size_um"]
    cells_fov_ids    = dataset_geo["cells_fov_ids"]

    rng = np.random.default_rng(SEED)
    candidates = list(cells_fov_ids)
    rng.shuffle(candidates)
    anchors = candidates[:N_ANCHORS]

    dfs = {}
    for channel in CHANNELS:
        corr_csv = cache_dir / f"correspondences_{channel}.csv"
        if not FORCE_RECOMPUTE and corr_csv.exists():
            df = pd.read_csv(corr_csv)
            print(f"[{dataset_geo['dataset_label']}][{channel}] Loaded {len(df)} cached correspondences from {corr_csv}")
        else:
            load_frame = load_frame_for_channel(channel)
            reporter = ProgressReporter(total=len(anchors) * 4,
                                         label=f"[{dataset_geo['dataset_label']}][{channel}] Registering edges")
            rows = []
            for batch_start in range(0, len(anchors), ANCHOR_BATCH_SIZE):
                for anchor_fov in anchors[batch_start:batch_start + ANCHOR_BATCH_SIZE]:
                    anchor_img = load_frame(anchor_fov)
                    for direction in ("up", "down", "left", "right"):
                        neighbor_fov = find_grid_neighbor(anchor_fov, full_positions, direction,
                                                           step_size_um, TOLERANCE_FRACTION)
                        reporter.update(1)
                        if neighbor_fov is None:
                            continue
                        neighbor_img = load_frame(neighbor_fov)
                        nominal_xy = full_positions[neighbor_fov]
                        measured_xy, error = register_neighbor_pair(
                            anchor_img, neighbor_img, full_positions[anchor_fov], nominal_xy,
                            direction, overlap_fraction, pixel_size_um, UPSAMPLE_FACTOR,
                        )
                        achieved_shift_um = (measured_xy[0] - nominal_xy[0], measured_xy[1] - nominal_xy[1])
                        achieved_corr = overlap_correlation(anchor_img, neighbor_img, direction, overlap_fraction,
                                                             extra_shift_um=achieved_shift_um, pixel_size_um=pixel_size_um)
                        nominal_corr  = overlap_correlation(anchor_img, neighbor_img, direction, overlap_fraction,
                                                             extra_shift_um=(0.0, 0.0), pixel_size_um=pixel_size_um)
                        rows.append({
                            "anchor_fov": anchor_fov, "neighbor_fov": neighbor_fov, "direction": direction,
                            "nominal_x": nominal_xy[0], "nominal_y": nominal_xy[1],
                            "measured_x": measured_xy[0], "measured_y": measured_xy[1],
                            "shift_um": float(np.hypot(*achieved_shift_um)),
                            "error": error, "achieved_correlation": achieved_corr, "nominal_correlation": nominal_corr,
                        })
                raw_cache.clear()
                gc.collect()
            reporter.done()

            df = pd.DataFrame(rows)
            df.to_csv(corr_csv, index=False)
            print(f"[{dataset_geo['dataset_label']}][{channel}] Saved {len(df)} correspondences to {corr_csv}")
        df["channel"] = channel
        dfs[channel] = df
    return dfs


for dataset_label, R in RESULTS.items():
    R["correspondence_dfs"] = compute_correspondences(R["dataset_geo"], R["load_frame_for_channel"], R["raw_cache"])
    total = sum(len(d) for d in R["correspondence_dfs"].values())
    print(f"[{dataset_label}] Total correspondence rows: {total}")

## 12 — Achieved correlation + measured-shift comparison (display, decisive)

In [ ]:
fig, axes = plt.subplots(len(DATASETS), 2, figsize=(12, 5 * len(DATASETS)), squeeze=False)
for row, (dataset_label, R) in enumerate(RESULTS.items()):
    dfs = R["correspondence_dfs"]

    ax = axes[row, 0]
    data_corr = [dfs[c]["achieved_correlation"].values for c in CHANNELS]
    ax.boxplot(data_corr, tick_labels=CHANNELS)
    ax.set_ylabel("overlap_correlation at achieved alignment", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{dataset_label}: achieved registration confidence", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

    ax = axes[row, 1]
    data_shift = [dfs[c]["shift_um"].values for c in CHANNELS]
    ax.boxplot(data_shift, tick_labels=CHANNELS)
    ax.set_ylabel("measured - nominal position (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{dataset_label}: measured shift magnitude", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.tight_layout()
fig.savefig(RESULTS["disk"]["dataset_geo"]["figures_dir"] / f"{NOTEBOOK_NAME}.achieved_correlation_and_shift.png", dpi=150)
plt.show()

print("Summary (mean +/- std):")
for dataset_label, R in RESULTS.items():
    print(f"[{dataset_label}]")
    for channel in CHANNELS:
        d = R["correspondence_dfs"][channel]
        print(f"  {channel:10s}: achieved_corr={d['achieved_correlation'].mean():.4f}+/-{d['achieved_correlation'].std():.4f}  "
              f"nominal_corr={d['nominal_correlation'].mean():.4f}+/-{d['nominal_correlation'].std():.4f}  "
              f"shift_um={d['shift_um'].mean():.3f}+/-{d['shift_um'].std():.3f}  (n={len(d)})")

## 13 — Closing the loop: DoG on a real border crop, raw vs. FFC'd

The actual picture that prompted this investigation -- does FFC visibly
improve the DoG-filtered edge clarity at the border?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
dataset_geo = RESULTS["disk"]["dataset_geo"]
load_frame_for_channel = RESULTS["disk"]["load_frame_for_channel"]
overlap_fraction = dataset_geo["overlap_fraction"]
sample_fov = int(RESULTS["disk"]["self_shift_df"]["fov_id"].iloc[0])

for ax, channel in zip(axes, CHANNELS):
    border_crop = crop_overlap(load_frame_for_channel(channel)(sample_fov), load_frame_for_channel(channel)(sample_fov),
                                BORDER_DIRECTION, overlap_fraction)[0]
    cleaned = remove_hot_pixels(border_crop).astype(np.float64)
    dog = difference_of_gaussians(cleaned, low_sigma=DOG_LOW_SIGMA, high_sigma=DOG_HIGH_SIGMA)
    ax.imshow(dog, cmap="gray")
    ax.set_title(f"disk: DoG({channel}), {BORDER_DIRECTION} border crop, FOV {sample_fov}", fontsize=PLOT_TITLE_FONTSIZE)
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
fig.savefig(dataset_geo["figures_dir"] / f"{NOTEBOOK_NAME}.dog_border_crop_raw_vs_ffc.png", dpi=150)
plt.show()

## 14 — Discussion

**Real result: FFC genuinely, partially recovers gradient energy toward
the frame border -- but that recovered energy does NOT translate into
better real registration, and on the two tests immune to a noise-
amplification confound, FFC-corrected DAPI performs slightly WORSE than
raw DAPI. Don't adopt FFC for stitching registration based on this
result.**

- **Section 6 radial gradient-energy test**: FFC substantially narrows
  the center-to-edge gap -- the edge/center ratio goes from 0.099 (raw)
  to 0.375 (FFC'd), a real ~3.8x recovery. This confirms vignetting is a
  genuine, partially FFC-fixable contributor to the original observation
  (clear center, unclear border in the DoG image). But the recovery is
  far from complete -- even FFC'd, gradient energy still falls ~3x from
  center to the frame's outer radius (Section 6's figure) -- some real,
  non-vignetting degradation clearly remains too.
- **Section 8 border-crop self-shift test (illustrative only, per this
  notebook's own caveat)**: on this real FOV, the border-crop curve is
  barely distinguishable in sharpness from the whole-frame curve, and (if
  anything) `dapi_raw` sits slightly ABOVE `dapi_ffc` at every shift in
  both regions -- FFC did not produce an artificially inflated
  self-shift peak here, but per the intro's caveat this single-FOV check
  is not proof it never would elsewhere.
- **Section 10 cross-tile test (decisive -- two independently-acquired
  real images can't share the same noise, so amplified noise can't
  inflate this metric)**: `dapi_ffc` is WORSE, not better -- lower
  absolute correlation (`corr(0)`: 0.312 vs. 0.432) AND less shift
  sensitivity (0-to-1px drop: 0.0023 vs. 0.0035). The gradient energy
  Section 6 found FFC recovering does not show up as real, correlatable
  structure between two separately-acquired tiles.
- **Section 12 practical whole-grid test (decisive, production-like)**:
  same direction -- `dapi_ffc`'s own end-to-end registration achieves
  LOWER achieved correlation confidence (0.412 vs. 0.488) and lower
  nominal (pre-registration) correlation (0.318 vs. 0.416), with
  comparable (not better) measured-shift noise. FFC does not improve, and
  mildly hurts, real registration outcomes on this dataset.
- **Section 13 -- closing the loop on the actual picture that started
  this investigation**: DoG applied to a raw vs. FFC'd border crop are
  visually indistinguishable. This makes sense given the two filters'
  length scales: this repo's own FFC field is smoothed at `sigma=50px`,
  while the DoG filter's own `high_sigma=8px` already subtracts out
  smooth local background at a much finer scale, well within a single
  overlap-band crop -- DoG is, locally, already doing most of what a
  smoothed FFC field would do. Combining the two here is close to
  redundant.

**Interpretation, tying back to `prompt_history/2026_09_04_1347_ffc_
border_registration_discussion.md`'s original two hypotheses**: this is
most consistent with a MIX where vignetting explains only PART of the
picture, and the FFC-recoverable portion is dominated by amplified noise
rather than real, usable structure (dividing by a field <1 amplifies
noise variance faster than it restores true signal) -- while a real,
FFC-uncorrectable residual (most plausibly off-axis optical aberration,
genuinely blurring the PSF, not just dimming it) likely accounts for the
remaining, unrecovered gradient-energy falloff in Section 6 and for why
neither decisive test improved.

**Bottom line**: don't add FFC to the stitching/registration pipeline
based on this result -- it visibly helps a single-image brightness/
gradient-energy diagnostic, but provides no benefit (and a small real
cost) on both tests that use two genuinely independent real acquisitions,
which is what stitching registration actually is. If pursued further:
directly quantify the aberration-vs-vignetting split (e.g. compare FFC'd
DAPI to a genuinely deconvolved/PSF-corrected frame) rather than treating
this as settled.